# Tawasil ESL: Level 1 proof of concept (landmark extraction)

This notebook tests the first steps of the Tawasil ESL pipeline (steps 2 and 3 of Figure 4.2 in the report) on real phone videos:

1. **Video input**: a 1-60 second video recorded on a phone (FR 4.1).
2. **Frame preprocessing (OpenCV)**: same orientation, a fixed size and 25 frames per second.
3. **Landmark extraction (MediaPipe)**: 21 points on each hand, 6 upper-body points and 45 face points (eyebrows, eyes, mouth) per frame, which is **93 points x 3 values = 279 numbers per frame**.

It does **not** recognize signs or write Arabic sentences yet. That is planned for SGP2.

**How to use it**
1. `Runtime` > `Run all`.
2. When the upload button appears, choose 1 to 5 short videos, **or one .zip file of frame folders** (for example Isharah samples, one folder per sentence).
3. At the end, a zip file with the results downloads automatically.

**Privacy:** use only videos of team members who agreed. The notebook deletes the uploaded videos at the end (like FR 4.6 in the app).

## Step 1: Install the libraries and download the MediaPipe models

In [ ]:
!pip install -q mediapipe==0.10.32
!wget -q -O hand_landmarker.task https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task
!wget -q -O pose_landmarker_lite.task https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/latest/pose_landmarker_lite.task
!wget -q -O face_landmarker.task https://storage.googleapis.com/mediapipe-models/face_landmarker/face_landmarker/float16/latest/face_landmarker.task
!ls -la *.task

## Step 2: Settings and helper functions

In [ ]:
import os, re, time, json, shutil, platform
import numpy as np
import pandas as pd
import cv2
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision

TARGET_FPS = 25          # Section 3.4: every video becomes 25 frames per second
LONG_SIDE = 640          # fixed size: the longer side of each frame becomes 640 pixels
MIN_SEC, MAX_SEC = 1, 60 # FR 4.1: videos of 1-60 seconds
IMG_EXT = (".jpg", ".jpeg", ".png", ".bmp")

# Points kept from each MediaPipe model
POSE_IDS = [11, 12, 13, 14, 15, 16]   # shoulders, elbows, wrists
FACE_IDS = [70, 63, 105, 66, 107, 55, 65, 52, 53, 46,          # left eyebrow (10)
            300, 293, 334, 296, 336, 285, 295, 282, 283, 276,  # right eyebrow (10)
            33, 133, 159, 145, 160, 144, 158, 153,             # left eye (8)
            362, 263, 386, 374, 387, 373, 385, 380,            # right eye (8)
            61, 291, 0, 17, 13, 14, 78, 308, 82]               # mouth (9)
N_POINTS = 21 + 21 + len(POSE_IDS) + len(FACE_IDS)            # 93 points
VALUES_PER_FRAME = N_POINTS * 3                               # x, y, z for each point
print("Points per frame:", N_POINTS, "| values per frame:", VALUES_PER_FRAME)

HAND_LINKS = [(0,1),(1,2),(2,3),(3,4),(0,5),(5,6),(6,7),(7,8),(5,9),(9,10),(10,11),(11,12),
              (9,13),(13,14),(14,15),(15,16),(13,17),(0,17),(17,18),(18,19),(19,20)]
POSE_LINKS = [(11,12),(11,13),(13,15),(12,14),(14,16)]

def make_detectors():
    """Create the three MediaPipe detectors in VIDEO mode."""
    B = lambda f: mp_python.BaseOptions(model_asset_path=f)
    V = vision.RunningMode.VIDEO
    hands = vision.HandLandmarker.create_from_options(
        vision.HandLandmarkerOptions(base_options=B("hand_landmarker.task"), running_mode=V, num_hands=2))
    pose = vision.PoseLandmarker.create_from_options(
        vision.PoseLandmarkerOptions(base_options=B("pose_landmarker_lite.task"), running_mode=V, num_poses=1))
    face = vision.FaceLandmarker.create_from_options(
        vision.FaceLandmarkerOptions(base_options=B("face_landmarker.task"), running_mode=V, num_faces=1))
    return hands, pose, face

def natural_key(f):
    return [int(t) if t.isdigit() else t.lower() for t in re.split(r"(\d+)", f)]

def preprocess_folder(path):
    """Frame folders (Isharah samples): the frame rate is not given, so every frame is kept and only resized."""
    names = sorted([f for f in os.listdir(path) if f.lower().endswith(IMG_EXT)], key=natural_key)
    raw = [cv2.imread(os.path.join(path, f)) for f in names]
    raw = [f for f in raw if f is not None]
    if not raw:
        return None, {"error": "no readable frames"}
    h, w = raw[0].shape[:2]
    scale = LONG_SIDE / max(h, w)
    size = (int(round(w * scale)), int(round(h * scale)))
    frames = [cv2.resize(f, size, interpolation=cv2.INTER_AREA) for f in raw]
    info = {"original_fps": "not given (frames)", "original_frames": len(raw), "duration_s": None,
            "original_size": f"{w}x{h}", "processed_size": f"{size[0]}x{size[1]}", "processed_frames": len(frames)}
    return frames, info

def preprocess(path):
    """OpenCV step: read the video upright, keep 25 fps and resize to a fixed size."""
    if os.path.isdir(path):
        return preprocess_folder(path)
    cap = cv2.VideoCapture(path)
    cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1)      # use the phone's rotation information
    src_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    raw = []
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        raw.append(frame)
    cap.release()
    n_src = len(raw)
    duration = n_src / src_fps if src_fps else 0
    if n_src == 0:
        return None, {"error": "could not read video"}
    h, w = raw[0].shape[:2]
    scale = LONG_SIDE / max(h, w)
    size = (int(round(w * scale)), int(round(h * scale)))
    n_out = max(1, int(round(duration * TARGET_FPS)))
    frames = []
    for i in range(n_out):                           # pick the source frame closest to each 25 fps time step
        src_idx = min(n_src - 1, int(round(i / TARGET_FPS * src_fps)))
        frames.append(cv2.resize(raw[src_idx], size, interpolation=cv2.INTER_AREA))
    info = {"original_fps": round(src_fps, 2), "original_frames": n_src,
            "duration_s": round(duration, 2), "original_size": f"{w}x{h}",
            "processed_size": f"{size[0]}x{size[1]}", "processed_frames": len(frames)}
    return frames, info

def to_xyz(landmarks, ids=None):
    pts = landmarks if ids is None else [landmarks[i] for i in ids]
    return np.array([[p.x, p.y, p.z] for p in pts], dtype=np.float32)

def extract(frames, detectors):
    """MediaPipe step: 279 numbers per frame (missing parts are filled with zeros)."""
    hands_d, pose_d, face_d = detectors
    out = np.zeros((len(frames), N_POINTS, 3), dtype=np.float32)
    found = {"left_hand": 0, "right_hand": 0, "pose": 0, "face": 0}
    times = []
    for i, frame in enumerate(frames):
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        img = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb)
        ts = int(i * 1000 / TARGET_FPS)
        t0 = time.perf_counter()
        hr, pr, fr = hands_d.detect_for_video(img, ts), pose_d.detect_for_video(img, ts), face_d.detect_for_video(img, ts)
        times.append((time.perf_counter() - t0) * 1000)
        for lms, hd in zip(hr.hand_landmarks, hr.handedness):
            slot = 0 if hd[0].category_name == "Left" else 21
            out[i, slot:slot + 21] = to_xyz(lms)
            found["left_hand" if slot == 0 else "right_hand"] += 1
        if pr.pose_landmarks:
            out[i, 42:42 + len(POSE_IDS)] = to_xyz(pr.pose_landmarks[0], POSE_IDS); found["pose"] += 1
        if fr.face_landmarks:
            out[i, 42 + len(POSE_IDS):] = to_xyz(fr.face_landmarks[0], FACE_IDS); found["face"] += 1
    return out, found, times

def draw(frame, pts):
    """Draw the extracted points on one frame (hands green, body blue, face red)."""
    img = frame.copy(); h, w = img.shape[:2]
    P = lambda p: (int(p[0] * w), int(p[1] * h))
    for start in (0, 21):
        hand = pts[start:start + 21]
        if np.any(hand):
            for a, b in HAND_LINKS: cv2.line(img, P(hand[a]), P(hand[b]), (0, 200, 0), 2)
            for p in hand: cv2.circle(img, P(p), 3, (0, 255, 0), -1)
    body = {pid: pts[42 + k] for k, pid in enumerate(POSE_IDS)}
    if any(np.any(v) for v in body.values()):
        for a, b in POSE_LINKS: cv2.line(img, P(body[a]), P(body[b]), (255, 120, 0), 3)
        for p in body.values(): cv2.circle(img, P(p), 5, (255, 120, 0), -1)
    for p in pts[42 + len(POSE_IDS):]:
        if np.any(p): cv2.circle(img, P(p), 2, (0, 0, 255), -1)
    return img

## Step 3: Upload your videos

In [ ]:
from google.colab import files
uploaded = files.upload()          # videos, or a .zip of frame folders (one folder = one sentence)
videos = []
for name in uploaded:
    if name.lower().endswith(".zip"):
        out_dir = "frames_" + os.path.splitext(name)[0]
        shutil.unpack_archive(name, out_dir)
        os.remove(name)
        for root, dirs, fs in os.walk(out_dir):
            if "__MACOSX" in root: continue
            if sum(f.lower().endswith(IMG_EXT) for f in fs) >= 5:
                videos.append(root)               # each folder of frames counts as one video
    else:
        videos.append(name)
print("To process:", videos)

## Step 3b (only if you uploaded dictionary word clips): join the clips into sentence videos

Name the clips before uploading as **S<sentence>_<word order>**, for example:
`S1_1.mp4` (I), `S1_2.mp4` (WANT), `S1_3.mp4` (WATER), then `S2_1.mp4`, `S2_2.mp4` ...

This cell joins each group into one video (`sentence_1.mp4`, `sentence_2.mp4` ...), all at the same size
and 25 fps. Files with other names are kept as they are. Source of the clips: Zayed Authority for People
of Determination, Sign Language Dictionary (used here for an academic test only).

In [ ]:
import re
from collections import defaultdict
groups, others = defaultdict(list), []
for name in videos:
    m = re.match(r"^[Ss](\d+)_(\d+)", os.path.basename(name))
    if m: groups[int(m.group(1))].append((int(m.group(2)), name))
    else: others.append(name)

def read_frames(path):
    cap = cv2.VideoCapture(path); cap.set(cv2.CAP_PROP_ORIENTATION_AUTO, 1)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0; fr = []
    while True:
        ok, f = cap.read()
        if not ok: break
        fr.append(f)
    cap.release()
    return fr, fps

sentence_videos = []
for sid in sorted(groups):
    clips = [n for _, n in sorted(groups[sid])]
    out_name, writer, size = f"sentence_{sid}.mp4", None, None
    for clip in clips:
        fr, fps = read_frames(clip)
        if not fr: print("Could not read", clip); continue
        if size is None:
            h, w = fr[0].shape[:2]; size = (w, h)
            writer = cv2.VideoWriter(out_name, cv2.VideoWriter_fourcc(*"mp4v"), TARGET_FPS, size)
        n_out = max(1, int(round(len(fr) / fps * TARGET_FPS)))
        for i in range(n_out):
            f = fr[min(len(fr) - 1, int(round(i / TARGET_FPS * fps)))]
            writer.write(cv2.resize(f, size, interpolation=cv2.INTER_AREA))
    if writer:
        writer.release(); sentence_videos.append(out_name)
        print(f"{out_name}: joined {len(clips)} clips -> {clips}")

if groups:
    joined_from = {f"sentence_{sid}.mp4": [n for _, n in sorted(groups[sid])] for sid in groups}
    with open("joined_clips.json", "w") as f: json.dump(joined_from, f, indent=2)
    for sid in groups:                      # the word clips are not needed any more
        for _, n in groups[sid]:
            if os.path.exists(n): os.remove(n)
    videos = sentence_videos + others
print("Videos to process:", videos)

## Step 4: Run the pipeline on each video

In [ ]:
os.makedirs("results", exist_ok=True)
rows = []
for name in videos:
    base = os.path.splitext(os.path.basename(os.path.normpath(name)))[0].replace(" ", "_")
    frames, info = preprocess(name)
    if frames is None:
        print(name, info); continue
    if info["duration_s"] is not None and not (MIN_SEC <= info["duration_s"] <= MAX_SEC):
        print(f"{name}: {info['duration_s']} s is outside 1-60 s, skipped (FR 4.1)"); continue
    detectors = make_detectors()                       # new detectors for each video (VIDEO mode timestamps)
    t0 = time.perf_counter()
    pts, found, times = extract(frames, detectors)
    total = time.perf_counter() - t0
    for d in detectors: d.close()
    n = len(frames)
    np.save(f"results/{base}_landmarks.npy", pts.reshape(n, -1))       # frames x 279 table
    mid = n // 2                                                       # example frame with the points drawn
    cv2.imwrite(f"results/{base}_frame_{mid}.png", np.hstack([frames[mid], draw(frames[mid], pts[mid])]))
    # privacy version for the report: the points only, on a black background (no face image)
    cv2.imwrite(f"results/{base}_frame_{mid}_points_only.png", draw(np.zeros_like(frames[mid]), pts[mid]))
    h, w = frames[0].shape[:2]                                         # short video with the points drawn
    vw = cv2.VideoWriter(f"results/{base}_landmarks.mp4", cv2.VideoWriter_fourcc(*"mp4v"), TARGET_FPS, (w, h))
    for f, p in zip(frames, pts): vw.write(draw(f, p))
    vw.release()
    rows.append({"video": name, **info,
                 "values_per_frame": pts.reshape(n, -1).shape[1],
                 "table_shape": f"{n} x {pts.reshape(n, -1).shape[1]}",
                 "hand_detected_%": round(100 * sum(np.any(pts[i, :42]) for i in range(n)) / n, 1),
                 "both_hands_%": round(100 * sum(np.any(pts[i, :21]) and np.any(pts[i, 21:42]) for i in range(n)) / n, 1),
                 "pose_detected_%": round(100 * found["pose"] / n, 1),
                 "face_detected_%": round(100 * found["face"] / n, 1),
                 "avg_ms_per_frame": round(float(np.mean(times)), 1),
                 "total_landmark_time_s": round(total, 2)})
    print(f"Done: {name}")

summary = pd.DataFrame(rows)
summary.to_csv("results/summary.csv", index=False)
with open("results/environment.json", "w") as f:
    json.dump({"python": platform.python_version(), "mediapipe": mp.__version__, "opencv": cv2.__version__,
               "machine": platform.processor() or platform.machine(),
               "note": "Times are measured on Google Colab, not on an Android phone."}, f, indent=2)
summary.T

## Step 5: Look at one example frame (left: preprocessed frame, right: extracted points)

In [ ]:
from IPython.display import Image as ShowImage, display
import glob
for p in sorted(glob.glob("results/*_frame_*.png")):
    print(p); display(ShowImage(p, width=700))

## Step 6: Delete the uploaded videos and download the results

In [ ]:
for name in videos:                     # privacy: remove the original videos and frames (like FR 4.6)
    if os.path.isdir(name): shutil.rmtree(name, ignore_errors=True)
    elif os.path.exists(name): os.remove(name)
for d in [d for d in os.listdir(".") if d.startswith("frames_")]:
    shutil.rmtree(d, ignore_errors=True)
if os.path.exists("joined_clips.json"): shutil.copy("joined_clips.json", "results/")
for v in videos:                        # also delete the joined sentence videos
    if os.path.exists(v): os.remove(v)
shutil.make_archive("tawasil_level1_results", "zip", "results")
files.download("tawasil_level1_results.zip")
print("Original videos deleted. Send tawasil_level1_results.zip for Chapter 7. Use the *_points_only.png images in the report (no face shown).")